# 03 — Submission Document Classification

## 1. Import Library

In [1]:
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import convnext_tiny
from tqdm.auto import tqdm

## 2. Konfigurasi Dataset dan Checkpoint

In [2]:
DATA_DIR = Path(
    "/kaggle/input/competitions/document-image-classification-sd234504"
)
OUTPUT_DIR = Path("/kaggle/working")
CHECKPOINT_NAME = "best_convnext_tiny_320.pth"

IMAGE_SIZE = 320
NUM_CLASSES = 16
BATCH_SIZE = 8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for csv_name in ("test.csv", "sample_submission.csv"):
    if not (DATA_DIR / csv_name).is_file():
        raise FileNotFoundError(
            f"{csv_name} tidak ditemukan pada {DATA_DIR}. "
            "Tambahkan dataset kompetisi melalui Add Input."
        )

test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_df = pd.read_csv(DATA_DIR / "sample_submission.csv")

# Utamakan checkpoint dari output Notebook 02B yang telah ditambahkan lewat Add Input.
checkpoint_candidates = sorted(
    Path("/kaggle/input").rglob(CHECKPOINT_NAME)
)
local_checkpoint = (
    OUTPUT_DIR / "convnext_320_experiment" / CHECKPOINT_NAME
)

if len(checkpoint_candidates) == 1:
    CHECKPOINT_PATH = checkpoint_candidates[0]
elif len(checkpoint_candidates) > 1:
    print("Ditemukan beberapa checkpoint:")
    for candidate in checkpoint_candidates:
        print(" -", candidate)
    raise RuntimeError(
        "Ada beberapa checkpoint bernama sama. "
        "Hapus input checkpoint yang tidak diperlukan atau isi CHECKPOINT_PATH "
        "secara eksplisit dengan path checkpoint ConvNeXt-Tiny 320 terbaik."
    )
elif local_checkpoint.is_file():
    CHECKPOINT_PATH = local_checkpoint
else:
    raise FileNotFoundError(
        "Checkpoint ConvNeXt-Tiny 320 belum ditemukan. "
        "Di Kaggle klik Add Input > Notebook Output > "
        "pilih output Notebook 02B versi yang menyimpan checkpoint "
        "best_convnext_tiny_320.pth (misalnya Version #3). "
        "Setelah itu jalankan ulang notebook ini."
    )

assert len(test_df) == len(sample_df) == 7994, "Jumlah data test tidak sesuai."
assert {"id", "filename"}.issubset(test_df.columns)
assert list(sample_df.columns) == ["id", "label"]
assert test_df["id"].tolist() == sample_df["id"].tolist(), (
    "Urutan ID test.csv dan sample_submission.csv berbeda."
)

print("Dataset     :", DATA_DIR)
print("Jumlah test :", len(test_df))
print("Checkpoint  :", CHECKPOINT_PATH)
print("Device      :", DEVICE)

Dataset     : /kaggle/input/competitions/document-image-classification-sd234504
Jumlah test : 7994
Checkpoint  : /kaggle/input/notebooks/aqilaikhza/02b-optimasi-document-classification/convnext_320_experiment/best_convnext_tiny_320.pth
Device      : cuda


## 3. Preprocessing dan DataLoader

In [3]:
class ResizeWithPadding:
    """Ubah ukuran gambar tanpa mengubah rasio aspek, dengan padding putih."""

    def __init__(self, size):
        self.size = size

    def __call__(self, image):
        image = image.convert("RGB")
        resized = ImageOps.contain(
            image,
            (self.size, self.size),
            method=Image.Resampling.BILINEAR,
        )
        canvas = Image.new("RGB", (self.size, self.size), (255, 255, 255))
        x = (self.size - resized.width) // 2
        y = (self.size - resized.height) // 2
        canvas.paste(resized, (x, y))
        return canvas


test_transform = transforms.Compose([
    ResizeWithPadding(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225],
    ),
])


class TestDocumentDataset(Dataset):
    """Memuat gambar TIFF dari metadata test tanpa label."""

    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = DATA_DIR / "test" / row["filename"]
        if not image_path.is_file():
            raise FileNotFoundError(f"Gambar test tidak ditemukan: {image_path}")
        with Image.open(image_path) as image:
            return self.transform(image.convert("RGB"))


test_dataset = TestDocumentDataset(test_df, test_transform)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
print("DataLoader siap:", len(test_dataset), "gambar")

DataLoader siap: 7994 gambar


## 4. Memuat Model ConvNeXt-Tiny 320 × 320

In [5]:
model = convnext_tiny(weights=None)
model.classifier[2] = nn.Linear(
    model.classifier[2].in_features,
    NUM_CLASSES,
)

# Checkpoint Notebook 02B menyimpan state_dict model.
state_dict = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True,
)
model.load_state_dict(state_dict, strict=True)
model = model.to(DEVICE)
model.eval()

print("Model         : ConvNeXt-Tiny")
print("Input size    :", f"{IMAGE_SIZE} × {IMAGE_SIZE}")
print("Jumlah kelas  :", NUM_CLASSES)
print("Checkpoint    :", CHECKPOINT_PATH)

Model         : ConvNeXt-Tiny
Input size    : 320 × 320
Jumlah kelas  : 16
Checkpoint    : /kaggle/input/notebooks/aqilaikhza/02b-optimasi-document-classification/convnext_320_experiment/best_convnext_tiny_320.pth


## 5. Inferensi pada Data Testing

In [6]:
predictions = []

with torch.inference_mode():
    for images in tqdm(test_loader, desc="Predicting Test Images"):
        images = images.to(DEVICE, non_blocking=True)
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda"),
        ):
            outputs = model(images)
        predictions.extend(outputs.argmax(dim=1).cpu().tolist())

assert len(predictions) == len(test_df), "Jumlah prediksi tidak sesuai."
assert all(0 <= int(label) < NUM_CLASSES for label in predictions)
print("Prediksi selesai:", len(predictions), "gambar")

Predicting Test Images:   0%|          | 0/1000 [00:00<?, ?it/s]

Prediksi selesai: 7994 gambar


## 6. Pembuatan dan Validasi Submission

In [7]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "label": predictions,
})

assert list(submission.columns) == ["id", "label"]
assert submission["id"].tolist() == sample_df["id"].tolist()
assert not submission.isna().any().any()
assert submission["label"].between(0, NUM_CLASSES - 1).all()

submission_path = OUTPUT_DIR / "submission.csv"
archive_path = OUTPUT_DIR / "submission_convnext_320.csv"
submission.to_csv(submission_path, index=False)
submission.to_csv(archive_path, index=False)

# Pastikan file yang ditulis sesuai format kompetisi.
check = pd.read_csv(submission_path)
assert check.shape == (7994, 2)
assert list(check.columns) == list(sample_df.columns)
assert check["id"].tolist() == sample_df["id"].tolist()
assert check["label"].between(0, 15).all()

print("SUBMISSION READY")
print("File utama    :", submission_path)
print("File arsip    :", archive_path)
print("Jumlah baris  :", len(check))
print("Jumlah kelas  :", check["label"].nunique())
print("\nPreview:")
print(check.head(10).to_string(index=False))
print("\nDistribusi label:")
print(check["label"].value_counts().sort_index().to_string())

SUBMISSION READY
File utama    : /kaggle/working/submission.csv
File arsip    : /kaggle/working/submission_convnext_320.csv
Jumlah baris  : 7994
Jumlah kelas  : 16

Preview:
        id  label
test_00001      0
test_00002      0
test_00003      6
test_00004      4
test_00005     13
test_00006      1
test_00007      9
test_00008     12
test_00009      8
test_00010     15

Distribusi label:
label
0     508
1     484
2     502
3     515
4     535
5     506
6     515
7     513
8     493
9     471
10    503
11    444
12    505
13    506
14    524
15    470
